## BioLORD + XGBoost with Hierarchical Chunking

Instead of embedding the full case summary into a single 768-dim vector (which dilutes discriminative signals), this notebook:

1. **Chunks** each clinical text into ~150-word segments with configurable overlap
2. **Embeds** each chunk independently with BioLORD-2023
3. **Aggregates** chunk embeddings per document using mean, max, and mean+max pooling
4. **Trains XGBoost** on each aggregation strategy and compares against the flat BioLORD baseline (notebook 02) and TF-IDF baseline (notebook 01)

### Imports

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import pickle
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, top_k_accuracy_score
)

In [2]:
import sys
sys.path.append('../')

In [3]:
from src.chunking import chunk_text, chunk_texts, aggregate_chunk_embeddings

### Configuration

In [4]:
CHUNK_SIZE = 150
OVERLAP = 20        # try 30 for comparison
MAX_CHUNKS = 15
MODEL_NAME = "FremyCompany/BioLORD-2023"
AGGREGATIONS = ["mean", "max", "mean_max"]

### Load classification split and chunk texts

In [5]:
# #colab only
# # Load classification split (text data needed for chunking)
# with open("/content/drive/MyDrive/Colab Notebooks/Projects/rare_disease_diagnosis/dataset/split_classification.pkl", "rb") as f:
#     X_train_clf, X_test_clf, y_train_clf, y_test_clf = pickle.load(f)

# print(f"Train: {len(X_train_clf)}  |  Test: {len(X_test_clf)}")
# print(f"Classes: {y_train_clf.nunique()}")

In [6]:
# Load classification split (text data needed for chunking)
with open("../data/split_classification.pkl", "rb") as f:
    X_train_clf, X_test_clf, y_train_clf, y_test_clf = pickle.load(f)

print(f"Train: {len(X_train_clf)}  |  Test: {len(X_test_clf)}")
print(f"Classes: {y_train_clf.nunique()}")

Train: 4312  |  Test: 1079
Classes: 320


In [7]:
# Chunk train and test texts
train_chunks, train_doc_lengths = chunk_texts(
    X_train_clf, chunk_size=CHUNK_SIZE, overlap=OVERLAP, max_chunks=MAX_CHUNKS
)
test_chunks, test_doc_lengths = chunk_texts(
    X_test_clf, chunk_size=CHUNK_SIZE, overlap=OVERLAP, max_chunks=MAX_CHUNKS
)

print(f"Train: {len(X_train_clf)} docs → {len(train_chunks)} chunks "
      f"(avg {len(train_chunks)/len(X_train_clf):.1f} chunks/doc)")
print(f"Test:  {len(X_test_clf)} docs → {len(test_chunks)} chunks "
      f"(avg {len(test_chunks)/len(X_test_clf):.1f} chunks/doc)")

Train: 4312 docs → 13390 chunks (avg 3.1 chunks/doc)
Test:  1079 docs → 3327 chunks (avg 3.1 chunks/doc)


In [8]:
# Distribution of chunks per document
from collections import Counter as C
print("Chunks/doc distribution (train):")
for n_chunks, count in sorted(C(train_doc_lengths).items()):
    print(f"  {n_chunks} chunk(s): {count} docs")

Chunks/doc distribution (train):
  1 chunk(s): 1244 docs
  2 chunk(s): 885 docs
  3 chunk(s): 658 docs
  4 chunk(s): 524 docs
  5 chunk(s): 405 docs
  6 chunk(s): 287 docs
  7 chunk(s): 138 docs
  8 chunk(s): 85 docs
  9 chunk(s): 39 docs
  10 chunk(s): 16 docs
  11 chunk(s): 8 docs
  12 chunk(s): 6 docs
  13 chunk(s): 5 docs
  14 chunk(s): 3 docs
  15 chunk(s): 9 docs


### Generate chunk embeddings (Colab)

Run this section on Colab with GPU to generate chunk-level embeddings. Once generated, save to Drive and copy to `data/`.

In [9]:
# !git clone https://github.com/amlopeza/rare_disease_diagnosis.git

In [10]:
# %cd rare_disease_diagnosis

In [11]:
# from google.colab import drive
# drive.mount('/content/drive')

In [12]:
# #── Colab only: generate chunk embeddings ──
# from src.embeddings import load_model, generate_embeddings

# model = load_model(MODEL_NAME)

# train_chunk_embs = generate_embeddings(train_chunks, model=model, batch_size=64)
# test_chunk_embs = generate_embeddings(test_chunks, model=model, batch_size=64)

# print(f"Train chunk embeddings: {train_chunk_embs.shape}")
# print(f"Test chunk embeddings:  {test_chunk_embs.shape}")

In [13]:
# # ── Colab only: save chunk embeddings to Drive ──
# import os

# BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Projects/rare_disease_diagnosis/dataset/"
# output_path = os.path.join(BASE_PATH, "embeddings_biolord_chunked_clf.pkl")

# with open(output_path, "wb") as f:
#     pickle.dump({
#         "train_chunk_embs": train_chunk_embs,
#         "test_chunk_embs": test_chunk_embs,
#         "train_doc_lengths": train_doc_lengths,
#         "test_doc_lengths": test_doc_lengths,
#         "y_train": y_train_clf,
#         "y_test": y_test_clf,
#         "config": {
#             "chunk_size": CHUNK_SIZE,
#             "overlap": OVERLAP,
#             "max_chunks": MAX_CHUNKS,
#             "model_name": MODEL_NAME,
#         }
#     }, f)

# print(f"Saved: {output_path}")

### Load pre-computed chunk embeddings (local)

In [14]:
# Load chunk embeddings
path = "../data/embeddings_biolord_chunked_clf.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

train_chunk_embs = data["train_chunk_embs"]
test_chunk_embs = data["test_chunk_embs"]
train_doc_lengths = data["train_doc_lengths"]
test_doc_lengths = data["test_doc_lengths"]
y_train = data["y_train"]
y_test = data["y_test"]
config = data["config"]

print(f"Config: {config}")
print(f"Train chunk embeddings: {train_chunk_embs.shape}")
print(f"Test chunk embeddings:  {test_chunk_embs.shape}")
print(f"Train docs: {len(train_doc_lengths)}  |  Test docs: {len(test_doc_lengths)}")

Config: {'chunk_size': 150, 'overlap': 20, 'max_chunks': 15, 'model_name': 'FremyCompany/BioLORD-2023'}
Train chunk embeddings: (13390, 768)
Test chunk embeddings:  (3327, 768)
Train docs: 4312  |  Test docs: 1079


### Aggregate chunk embeddings and encode labels

In [15]:
# Encode labels to contiguous 0..N-1 range for XGBoost
le = LabelEncoder()
le.fit(pd.concat([y_train, y_test]))
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

n_classes = len(le.classes_)
print(f"Classes: {n_classes}")

# Compute sample weights to handle class imbalance
class_counts = Counter(y_train_enc)
n_samples = len(y_train_enc)
n_cls = len(class_counts)

weight_map = {cls: n_samples / (n_cls * cnt)
              for cls, cnt in class_counts.items()}
sample_weights = np.array([weight_map[y] for y in y_train_enc])

print(f"Weight range: {sample_weights.min():.2f} - {sample_weights.max():.2f}")

Classes: 320
Weight range: 0.15 - 2.69


### XGBoost experiments per aggregation strategy

In [16]:
results = {}

for agg in AGGREGATIONS:
    print(f"\n{'='*50}")
    print(f"Aggregation: {agg}")
    print(f"{'='*50}")

    # Aggregate chunk embeddings to document level
    X_train = aggregate_chunk_embeddings(train_chunk_embs, train_doc_lengths, method=agg)
    X_test = aggregate_chunk_embeddings(test_chunk_embs, test_doc_lengths, method=agg)

    n_features = X_train.shape[1]
    print(f"Feature dim: {n_features}")

    num_class = n_classes
    xgb_clf = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist",
        eval_metric="mlogloss",
        early_stopping_rounds=30,
        verbosity=1,
    )

    xgb_clf.fit(
        X_train, y_train_enc,
        sample_weight=sample_weights,
        eval_set=[(X_test, y_test_enc)],
        verbose=50,
    )

    # Evaluate
    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)

    acc = accuracy_score(y_test_enc, y_pred)
    f1_macro = f1_score(y_test_enc, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_test_enc, y_pred, average="weighted", zero_division=0)
    top3 = top_k_accuracy_score(y_test_enc, y_proba, k=3, labels=range(num_class))
    top5 = top_k_accuracy_score(y_test_enc, y_proba, k=5, labels=range(num_class))
    top10 = top_k_accuracy_score(y_test_enc, y_proba, k=10, labels=range(num_class))

    results[agg] = {
        "Accuracy": acc,
        "F1 (macro)": f1_macro,
        "F1 (weighted)": f1_weighted,
        "Top-3": top3,
        "Top-5": top5,
        "Top-10": top10,
    }

    print(f"\nAccuracy:       {acc:.4f}")
    print(f"F1 (macro):     {f1_macro:.4f}")
    print(f"F1 (weighted):  {f1_weighted:.4f}")
    print(f"Top-3:          {top3:.4f}")
    print(f"Top-5:          {top5:.4f}")
    print(f"Top-10:         {top10:.4f}")


Aggregation: mean
Feature dim: 768
[0]	validation_0-mlogloss:5.68048
[50]	validation_0-mlogloss:4.03340
[100]	validation_0-mlogloss:3.81094
[150]	validation_0-mlogloss:3.80078
[152]	validation_0-mlogloss:3.80204

Accuracy:       0.2345
F1 (macro):     0.1506
F1 (weighted):  0.2087
Top-3:          0.4069
Top-5:          0.4903
Top-10:         0.5968

Aggregation: max
Feature dim: 768
[0]	validation_0-mlogloss:5.70093
[50]	validation_0-mlogloss:4.22262
[100]	validation_0-mlogloss:3.97500
[150]	validation_0-mlogloss:3.93849
[200]	validation_0-mlogloss:3.93915
[214]	validation_0-mlogloss:3.94150

Accuracy:       0.2215
F1 (macro):     0.1402
F1 (weighted):  0.1984
Top-3:          0.3781
Top-5:          0.4671
Top-10:         0.5765

Aggregation: mean_max
Feature dim: 1536
[0]	validation_0-mlogloss:5.68137
[50]	validation_0-mlogloss:4.04967
[100]	validation_0-mlogloss:3.81083
[150]	validation_0-mlogloss:3.79980
[162]	validation_0-mlogloss:3.80142

Accuracy:       0.2428
F1 (macro):     0.1

### Results comparison

In [17]:
# Chunked results table
results_df = pd.DataFrame(results).T
results_df.index.name = "Aggregation"
print("Chunked BioLORD + XGBoost results:")
print(results_df.to_string(float_format="{:.4f}".format))
print()

Chunked BioLORD + XGBoost results:
             Accuracy  F1 (macro)  F1 (weighted)  Top-3  Top-5  Top-10
Aggregation                                                           
mean           0.2345      0.1506         0.2087 0.4069 0.4903  0.5968
max            0.2215      0.1402         0.1984 0.3781 0.4671  0.5765
mean_max       0.2428      0.1655         0.2187 0.4041 0.4986  0.6070



In [19]:
# Full comparison with baselines from notebooks 01 and 02
best_agg = max(results, key=lambda k: results[k]["F1 (macro)"])
best = results[best_agg]

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "F1 (macro)", "F1 (weighted)"],
    "TF-IDF + LogReg": [0.3160, 0.2909, 0.3057],
    "BioLORD flat + XGB": [0.1798, 0.1107, 0.1611],
    f"BioLORD chunked ({best_agg}) + XGB": [
        best["Accuracy"], best["F1 (macro)"], best["F1 (weighted)"]
    ],
})
comparison

,Metric,TF-IDF + LogReg,BioLORD flat + XGB,BioLORD chunked (mean_max) + XGB
0,Accuracy,0.3160,0.1798,0.242817
1,F1 (macro),0.2909,0.1107,0.165517
2,F1 (weighted),0.3057,0.1611,0.218725


In [20]:
# Classification report for best aggregation
best_X_test = aggregate_chunk_embeddings(test_chunk_embs, test_doc_lengths, method=best_agg)
best_X_train = aggregate_chunk_embeddings(train_chunk_embs, train_doc_lengths, method=best_agg)

# Retrain best model for detailed report
xgb_best = XGBClassifier(
    objective="multi:softprob",
    num_class=n_classes,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=4,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    verbosity=0,
)

xgb_best.fit(
    best_X_train, y_train_enc,
    sample_weight=sample_weights,
    eval_set=[(best_X_test, y_test_enc)],
    verbose=False,
)

y_pred_best = xgb_best.predict(best_X_test)
print(f"Classification report — BioLORD chunked ({best_agg}):\n")
print(classification_report(y_test_enc, y_pred_best, zero_division=0))

Classification report — BioLORD chunked (mean_max):

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.00      0.00      0.00         3
           2       0.20      0.33      0.25         6
           3       0.00      0.00      0.00         4
           4       0.00      0.00      0.00         6
           5       0.25      0.50      0.33         2
           6       0.00      0.00      0.00         3
           7       0.17      0.33      0.22         3
           8       0.25      0.17      0.20         6
           9       0.29      0.67      0.40         3
          10       0.00      0.00      0.00         2
          11       0.00      0.00      0.00         1
          12       0.00      0.00      0.00         3
          13       0.00      0.00      0.00         1
          14       0.00      0.00      0.00         1
          15       0.00      0.00      0.00         3
          16       0.33     

### Summary and Analysis

**Config:** BioLORD-2023, chunk_size=150, overlap=20, max_chunks=15. Average 3.1 chunks/doc (~29% of documents are single-chunk).

#### Results by aggregation strategy

| Aggregation | Accuracy | F1 (macro) | F1 (weighted) | Top-3 | Top-5 | Top-10 |
|---|---|---|---|---|---|---|
| mean | 0.2345 | 0.1506 | 0.2087 | 0.4069 | 0.4903 | 0.5968 |
| max | 0.2215 | 0.1402 | 0.1984 | 0.3781 | 0.4671 | 0.5765 |
| **mean_max** | **0.2428** | **0.1655** | **0.2187** | **0.4041** | **0.4986** | **0.6070** |

#### Comparison across all approaches (same 320-class split)

| Metric | TF-IDF + LogReg | BioLORD flat + XGB | BioLORD chunked (mean_max) + XGB |
|---|---|---|---|
| Accuracy | **0.3160** | 0.1798 | 0.2428 |
| F1 (macro) | **0.2909** | 0.1107 | 0.1655 |
| F1 (weighted) | **0.3057** | 0.1611 | 0.2187 |

#### Analysis

**Chunking improves over flat embeddings across the board.** Mean_max pooling (best strategy) improves over flat BioLORD by +6.3pp accuracy, +5.5pp F1 macro, and +5.8pp F1 weighted. Top-10 accuracy jumps from 45.8% to 60.7%, confirming that preserving chunk-level information helps the model locate the correct disease neighborhood.

**Mean_max outperforms both mean and max individually.** Concatenating both vectors (1536-dim) gives XGBoost access to complementary signals: mean captures the overall clinical context while max retains the strongest discriminative features from individual chunks. This combination provides more information than either alone.

**Max pooling alone performs worst.** This is likely because max pooling is noisier — a single chunk with an extreme activation in some dimension can dominate, regardless of whether that activation is clinically meaningful.

**TF-IDF + LogReg still wins by a significant margin.** Despite the improvement from chunking, TF-IDF maintains a ~7pp lead in accuracy and ~12pp in F1 macro. The core reasons identified in notebook 02 still apply:
- TF-IDF preserves exact lexical features (gene names, syndrome terms) that are highly discriminative for rare diseases
- Even with chunking, BioLORD still averages/pools token-level information within each 150-word chunk — specific terms get diluted, just less so
- Logistic Regression on sparse TF-IDF features can leverage many weak lexical signals simultaneously, while XGBoost on dense features relies on axis-aligned splits

**The gap is narrowing, which validates the chunking hypothesis.** The improvement from flat to chunked shows that information loss during pooling *was* a real problem. However, the embedding approach needs a fundamentally different downstream strategy — retrieval (notebook 05) rather than classification — to fully leverage the semantic structure that embeddings capture.

#### Next steps
- Test overlap=30 to see if more context sharing between chunks helps
- Proceed to retrieval-based evaluation (notebook 05) where embeddings should excel